In [ ]:
import pandas
import os
import matplotlib.pyplot as plt
from scipy import stats
from files_processor import FilesProcessor


In [ ]:
df_fare_metrics_by_year_deflated = pandas.read_csv("metrics_files/fare_metrics_by_year_deflated.csv")
df_fare_metrics_by_year = pandas.read_csv("metrics_files/fare_metrics_by_year.csv")
df_route_fare_variability = pandas.read_csv("metrics_files/route_fare_variability.csv")

### Média de rotas reportando as tarifas

antes vs depois de 2010

In [20]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats


metrics_a = pandas.read_csv(os.getcwd() + "/metrics_files/fare_metrics_by_year_deflated.csv", sep=",", encoding="utf-8")
metrics_a['Year'] = metrics_a['YearMonth'].str[:4].astype(int)

def weighted_average(values, weights):
    return (values * weights).sum() / weights.sum()

yearly = metrics_a[metrics_a['Year'] < 2026].groupby('Year').apply(lambda x: pd.Series({
    'AvgFareNominal': weighted_average(x['WeightedAverageFare'], x['TotalSeats']),
    'AvgFareReal': weighted_average(x['WeightedAverageFareReal'], x['TotalSeats']),
    'RoutesReporting': x['RouteAgg'].nunique(),
}), include_groups=False).reset_index()
yearly

,Year,AvgFareNominal,AvgFareReal,RoutesReporting
0,2002,271.863695,1091.597807,63.0
1,2003,344.736919,1207.442023,57.0
2,2004,386.021970,1267.751940,75.0
3,2005,395.520775,1219.354308,67.0
4,2006,376.994494,1115.610081,67.0
5,2007,286.604523,817.092686,68.0
6,2008,417.739408,1126.144987,66.0
7,2009,315.432904,812.693961,66.0
8,2010,272.465600,664.545130,2608.0
9,2011,269.871732,619.532328,3317.0


In [22]:
# média de rotas reportando antes de 2010
x = yearly[yearly['Year'] < 2010]['RoutesReporting']
print(x.mean())

# média de rotas reportando depois de 2010
y = yearly[yearly['Year'] >= 2010]['RoutesReporting']
print(y.mean())


66.125
4524.125


### Tabela de rotas mais frequentes 
+ representação do total

top 5 anual

In [9]:
import pandas
import os

metrics = pandas.read_csv(os.getcwd() + "/metrics_files/fare_metrics_by_year_deflated.csv", sep=",", encoding="utf-8")
metrics['Year'] = metrics['YearMonth'].str[:4].astype(int)
metrics = metrics[metrics['Year'] < 2026]

route_seats = metrics.groupby('RouteAgg')['TotalSeats'].sum().reset_index()

# adiciona coluna com representação do total em %
route_seats['TotalSeatsPercent'] = route_seats['TotalSeats'] / route_seats['TotalSeats'].sum() * 100
route_seats = route_seats.sort_values('TotalSeats', ascending=False)
route_seats['TotalSeatsPercentAcum'] = route_seats['TotalSeatsPercent'].cumsum()

# top5_routes = route_seats.head(5)['RouteAgg'].tolist()
route_seats.head(5)

,RouteAgg,TotalSeats,TotalSeatsPercent,TotalSeatsPercentAcum
1304,CGH >> SDU,41567331.0,5.714683,5.714683
534,BSB >> CGH,16380724.0,2.252024,7.966707
3024,GRU >> SSA,15577531.0,2.141601,10.108308
1223,CGH >> POA,13581038.0,1.867123,11.975431
2913,GRU >> REC,12698846.0,1.745839,13.721270


top 5 mensal

In [19]:
WINDOW_START, WINDOW_END = '2016-01', '2026-08'

monthly = pandas.read_csv(os.getcwd() + "/metrics_files/fare_metrics_by_year_deflated.csv", sep=",", encoding="utf-8")
monthly = monthly[(monthly['YearMonth'] >= WINDOW_START) & (monthly['YearMonth'] <= WINDOW_END)]
# print(f"Months in window: {monthly['YearMonth'].nunique()} ({monthly['YearMonth'].min()} to {monthly['YearMonth'].max()})")


route_seats_window = monthly.groupby('RouteAgg')['TotalSeats'].sum().reset_index().sort_values('TotalSeats', ascending=False)
route_seats_window['TotalSeatsPercent'] = route_seats_window['TotalSeats'] / route_seats_window['TotalSeats'].sum() * 100
route_seats_window['TotalSeatsPercentAcum'] = route_seats_window['TotalSeatsPercent'].cumsum()

# top5_routes_window = route_seats_window.head(5).index.tolist()
# print(f"Entering the top 5 vs. the 2002-2025 ranking: {sorted(set(top5_routes_window) - set(top5_routes))}")
# print(f"Dropping out of the top 5: {sorted(set(top5_routes) - set(top5_routes_window))}")
route_seats_window.head(5)

,RouteAgg,TotalSeats,TotalSeatsPercent,TotalSeatsPercentAcum
1255,CGH >> SDU,12671678.0,3.676682,3.676682
515,BSB >> CGH,6076945.0,1.763223,5.439905
2800,GRU >> REC,5694849.0,1.652358,7.092263
1153,CGH >> CNF,5470274.0,1.587198,8.679460
1179,CGH >> POA,5160268.0,1.497250,10.176710


### Gráfico top 5 vs resto | mensal de 2015 a 2026

In [ ]:


metrics = pandas.read_csv(os.getcwd() + "/metrics_files/fare_metrics_by_year_deflated.csv", sep=",", encoding="utf-8")
metrics['Year'] = metrics['YearMonth'].str[:4].astype(int)
# metrics = metrics[metrics['Year'] < 2026]

route_seats = metrics.groupby('RouteAgg')['TotalSeats'].sum().reset_index()
route_seats = route_seats.sort_values('TotalSeats', ascending=False)
top5_routes = route_seats.head(5)['RouteAgg'].tolist()
route_seats.head(5)


In [ ]:
top5_yearly = {}
for route in top5_routes:
    sub = metrics[metrics['RouteAgg'] == route]
    route_yearly = sub.groupby('Year').apply(
        lambda x: weighted_average(x['WeightedAverageFareReal'], x['TotalSeats']), include_groups=False
    ).rename('AvgFareReal').reset_index()
    top5_yearly[route] = route_yearly

    print(f"=== {route} ({sub['TotalSeats'].sum():,.0f} total seats, {len(route_yearly)} years of data) ===")
    trend_test(route_yearly, 'AvgFareReal', f'{route}, full period 2002-2025')
    trend_test(route_yearly[route_yearly['Year'] >= 2011], 'AvgFareReal', f'{route}, 2011-2025')


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for route in top5_routes:
    route_yearly = top5_yearly[route]
    ax.plot(route_yearly['Year'], route_yearly['AvgFareReal'], marker='o', label=route)
ax.axvline(2010.5, color='red', linestyle=':', alpha=0.5)
ax.text(2010.6, ax.get_ylim()[1] * 0.02 + ax.get_ylim()[0], 'Jul 2010: Res. 140/2010', fontsize=8, color='red')
ax.set_xlabel('Year')
ax.set_ylabel(f"Real average fare (BRL, {metrics['YearMonth'].max()} purchasing power)")
ax.set_title('Top 5 busiest routes: real (inflation-adjusted) fare, 2002-2025')
ax.legend()
plt.tight_layout()
plt.show()
